# Tratamiento de Datos — CyberGuard API + CVE Scraper

**Universidad Internacional del Ecuador — UIDE EIG**  
**Maestria en Business Intelligence | 2025**  
**Docente: Prof. Ivan Reyes**

---

Este notebook demuestra el tratamiento completo de datos del proyecto:

- Consumo en vivo de la CyberGuard API
- Carga y limpieza del dataset de vulnerabilidades CVE
- Analisis estadistico descriptivo
- Visualizaciones del dataset

---
## 1. Importacion de librerias

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

BASE_API = 'http://localhost:8000/api/v1'

print('Librerias importadas correctamente.')

ModuleNotFoundError: No module named 'pandas'

---
## 2. Verificacion del estado de la API

In [ ]:
r = requests.get('http://localhost:8000/health').json()

print('=' * 45)
print('  CYBERGUARD API - ESTADO DEL SERVICIO')
print('=' * 45)
print(f"  Estado:    {r['status']}")
print(f"  Version:   {r['version']}")
print(f"  Timestamp: {r['timestamp']}")
print('=' * 45)

---
## 3. Modulo Password — Analisis de fortaleza de contrasenas

In [ ]:
contrasenas = [
    '123456',
    'password',
    'uide2025',
    'Uide2025!',
    'X#9kL@mP2qR!vN8z'
]

resultados = []
for pwd in contrasenas:
    r = requests.post(f'{BASE_API}/password/analyze', json={'password': pwd}).json()
    resultados.append({
        'Contrasena': pwd[:8] + '...' if len(pwd) > 8 else pwd,
        'Score': int(r['score'].split('/')[0]),
        'Fortaleza': r['strength'],
        'Entropia': r['entropy_bits'],
        'Tiempo crackeo': r['crack_time_estimate']
    })

df_pwd = pd.DataFrame(resultados)
print(df_pwd.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Analisis de Fortaleza de Contrasenas', fontsize=13, fontweight='bold', y=1.01)

colores = ['#e74c3c' if s <= 3 else '#e67e22' if s <= 5 else '#2ecc71' for s in df_pwd['Score']]

axes[0].barh(df_pwd['Contrasena'], df_pwd['Score'], color=colores, edgecolor='white')
axes[0].set_xlabel('Score (0-10)')
axes[0].set_title('Score de Fortaleza')
axes[0].set_xlim(0, 10)
axes[0].axvline(x=7, color='gray', linestyle='--', alpha=0.5, label='Umbral seguro')
axes[0].legend(fontsize=8)

axes[1].barh(df_pwd['Contrasena'], df_pwd['Entropia'], color='#3498db', edgecolor='white')
axes[1].set_xlabel('Bits de entropia')
axes[1].set_title('Entropia de Shannon')

plt.tight_layout()
plt.savefig('grafica_passwords.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafica guardada como grafica_passwords.png')

---
## 4. Modulo URL — Deteccion de phishing

In [ ]:
urls_prueba = [
    'https://google.com',
    'https://uide.edu.ec/login',
    'http://paypal-secure-login.tk/verify-account',
    'http://192.168.1.1/admin@bank.com/update-payment',
    'https://bit.ly/confirm-identity'
]

resultados_url = []
for url in urls_prueba:
    r = requests.post(f'{BASE_API}/url/analyze', json={'url': url}).json()
    resultados_url.append({
        'URL': url[:45] + '...' if len(url) > 45 else url,
        'Risk Score': int(r['risk_score'].split('/')[0]),
        'Nivel': r['risk_level'],
        'HTTPS': r['has_https'],
        'Flags': len(r['suspicious_flags'])
    })

df_url = pd.DataFrame(resultados_url)
print(df_url.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colores_url = ['#e74c3c' if s >= 6 else '#e67e22' if s >= 3 else '#2ecc71' for s in df_url['Risk Score']]

bars = ax.barh(df_url['URL'], df_url['Risk Score'], color=colores_url, edgecolor='white')
ax.set_xlabel('Risk Score (0-10)')
ax.set_title('Analisis de Riesgo de URLs', fontweight='bold')
ax.set_xlim(0, 10)
ax.axvline(x=6, color='#e74c3c', linestyle='--', alpha=0.6, label='Alto riesgo')
ax.axvline(x=3, color='#e67e22', linestyle='--', alpha=0.6, label='Sospechoso')
ax.legend(fontsize=8)

for bar, score in zip(bars, df_url['Risk Score']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{score}/10', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('grafica_urls.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Modulo Hash — Generacion con multiples algoritmos

In [ ]:
r = requests.post(f'{BASE_API}/hash/generate', json={'text': 'CyberGuard UIDE 2025', 'algorithm': 'sha256'}).json()

print(f"Input:     {r['input']}")
print(f"Algoritmo: {r['algorithm']}")
print(f"Hash:      {r['hash']}")
print(f"Bytes:     {r['input_bytes']}")
print()
print('Todos los hashes generados:')
print('-' * 55)
for algo, h in r['all_hashes'].items():
    print(f"  {algo.upper():<12} {h[:40]}...")

---
## 6. Modulo Crypto — Generacion de claves seguras

In [ ]:
r = requests.get(f'{BASE_API}/crypto/keygen', params={'bits': 256}).json()

print('=' * 55)
print(f"  Bits:          {r['bits']}")
print(f"  Hex Key:       {r['hex_key'][:32]}...")
print(f"  Base64:        {r['base64_key'][:32]}...")
print(f"  API Key:       {r['api_key_format']}")
print(f"  UUID v4:       {r['uuid_v4_style']}")
print('=' * 55)

---
## 7. Carga del dataset CVE — NIST National Vulnerability Database

In [ ]:
data_dir = Path('../cve-scraper/data')
csv_files = sorted(data_dir.glob('cve_vulnerabilities_*.csv'))

if not csv_files:
    print('No se encontro el CSV. Ejecuta primero python scraper.py en cve-scraper/')
else:
    csv_path = csv_files[-1]
    df = pd.read_csv(csv_path, index_col=0)
    print(f'Dataset cargado: {csv_path.name}')
    print(f'Registros: {len(df)}')
    print(f'Columnas:  {len(df.columns)}')
    print()
    print(df[['cve_id', 'cvss_score', 'severity', 'attack_vector', 'published_date']].head(8).to_string())

---
## 8. Analisis estadistico descriptivo del dataset CVE

In [ ]:
print('=' * 50)
print('  ESTADISTICAS DESCRIPTIVAS — CVSS SCORE')
print('=' * 50)
print(df['cvss_score'].describe().round(2).to_string())
print()
print('Distribucion por severidad:')
print(df['severity'].value_counts().to_string())
print()
print('Distribucion por vector de ataque:')
print(df['attack_vector'].value_counts().to_string())

---
## 9. Visualizaciones del dataset CVE

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analisis de Vulnerabilidades CVE — NIST NVD', fontsize=14, fontweight='bold', y=1.01)

# Grafica 1 - Distribucion CVSS Score
axes[0, 0].hist(df['cvss_score'].dropna(), bins=15, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Distribucion de CVSS Score')
axes[0, 0].set_xlabel('Score')
axes[0, 0].set_ylabel('Cantidad de CVEs')
axes[0, 0].axvline(df['cvss_score'].mean(), color='black', linestyle='--', alpha=0.7,
                   label=f"Media: {df['cvss_score'].mean():.2f}")
axes[0, 0].legend(fontsize=9)

# Grafica 2 - Por Severidad
sev_counts = df['severity'].value_counts()
colores_sev = ['#c0392b', '#e67e22', '#f1c40f', '#2ecc71']
axes[0, 1].bar(sev_counts.index, sev_counts.values,
               color=colores_sev[:len(sev_counts)], edgecolor='white')
axes[0, 1].set_title('CVEs por Nivel de Severidad')
axes[0, 1].set_xlabel('Severidad')
axes[0, 1].set_ylabel('Cantidad')
for i, (idx, val) in enumerate(sev_counts.items()):
    axes[0, 1].text(i, val + 0.5, str(val), ha='center', fontsize=10)

# Grafica 3 - Vector de Ataque
vec_counts = df['attack_vector'].value_counts()
axes[1, 0].pie(vec_counts.values, labels=vec_counts.index, autopct='%1.1f%%',
               colors=['#3498db', '#e74c3c', '#2ecc71', '#e67e22'],
               startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1, 0].set_title('Distribucion por Vector de Ataque')

# Grafica 4 - Top 10 CWEs
cwes = df['cwe_ids'].dropna().str.split(' | ').explode()
cwes = cwes[cwes != 'N/A']
top_cwes = cwes.value_counts().head(8)
axes[1, 1].barh(top_cwes.index[::-1], top_cwes.values[::-1],
                color='#9b59b6', edgecolor='white')
axes[1, 1].set_title('Top 8 Tipos de Debilidad (CWE)')
axes[1, 1].set_xlabel('Ocurrencias')

plt.tight_layout()
plt.savefig('grafica_cve_analisis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafica guardada como grafica_cve_analisis.png')

---
## 10. Top 10 vulnerabilidades mas criticas

In [ ]:
top10 = df.nlargest(10, 'cvss_score')[['cve_id', 'cvss_score', 'severity', 'attack_vector', 'published_date', 'affected_products']]
top10['affected_products'] = top10['affected_products'].str[:40]

print('TOP 10 VULNERABILIDADES MAS CRITICAS')
print('=' * 90)
print(top10.to_string(index=False))

---
## 11. Exportacion del dataset procesado

In [ ]:
resumen = pd.DataFrame({
    'Metrica': [
        'Total CVEs analizados',
        'CVEs Criticos (>= 9.0)',
        'CVEs Alto riesgo (>= 7.0)',
        'CVSS Score promedio',
        'Score maximo',
        'Vector mas frecuente',
        'CWE mas comun'
    ],
    'Valor': [
        len(df),
        len(df[df['cvss_score'] >= 9.0]),
        len(df[df['cvss_score'] >= 7.0]),
        round(df['cvss_score'].mean(), 2),
        df['cvss_score'].max(),
        df['attack_vector'].value_counts().idxmax(),
        cwes.value_counts().idxmax() if not cwes.empty else 'N/A'
    ]
})

print('RESUMEN EJECUTIVO DEL DATASET')
print('=' * 40)
print(resumen.to_string(index=False))

resumen.to_csv('resumen_ejecutivo.csv', index=False)
print()
print('Resumen exportado como resumen_ejecutivo.csv')

---
## 12. Conclusiones

Este notebook demuestra la integracion completa entre los dos componentes del proyecto:

- **CyberGuard API** provee herramientas de analisis de seguridad en tiempo real accesibles mediante endpoints REST documentados.

- **CVE Scraper** extrae datos reales del NIST NVD y los transforma en un dataset estructurado listo para analisis de Business Intelligence.

- El tratamiento de datos incluye limpieza, deduplicacion, categorizacion, analisis estadistico descriptivo y visualizacion con matplotlib.

- Los resultados muestran que la mayoria de vulnerabilidades criticas se explotan a traves de la red (NETWORK), lo que refuerza la importancia de controles de seguridad perimetral en las organizaciones.